# ✅ Ejercicios resueltos: Regresión Logística

Este notebook contiene la **resolución paso a paso** de los 4 ejercicios de la sección 9 de la *Práctica guiada de Regresión Logística*.

> 💡 **Consejo:** intentá resolver cada ejercicio por tu cuenta **antes** de mirar la solución. Después comparala con la tuya: hay muchas formas correctas de resolver un mismo ejercicio.

Cada ejercicio incluye el enunciado, el código comentado y la interpretación de los resultados.

## Preparación: importar las librerías

Importamos todo lo que vamos a necesitar en los cuatro ejercicios. Si te falta alguna librería, ejecutá antes `%pip install numpy pandas matplotlib seaborn scikit-learn`.

In [ ]:
import numpy as np                                    # cálculos numéricos
import pandas as pd                                   # tablas de datos
import matplotlib.pyplot as plt                       # gráficos
import seaborn as sns                                 # gráficos estadísticos
from matplotlib import cm                             # paletas de colores

from sklearn.datasets import make_blobs, make_classification        # generadores de datasets sintéticos
from sklearn.linear_model import LogisticRegression                 # el modelo
from sklearn.model_selection import train_test_split                # dividir en train / test
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

sns.set_theme(style="whitegrid")                      # estilo de los gráficos
print("¡Listo! Librerías importadas.")

### Funciones auxiliares

Como vamos a repetir algunas tareas en varios ejercicios, definimos dos **funciones** (bloques de código reutilizables):

- **`sigmoid_function(x, w0, w1)`**: calcula la probabilidad de clase 1 con la fórmula de la sigmoide.
- **`graficar_matriz(y_real, y_pred, ax, titulo, etiquetas)`**: dibuja una matriz de confusión como mapa de calor en el gráfico `ax` que le indiquemos.

In [ ]:
def sigmoid_function(x, w0, w1):
    # Probabilidad de clase 1: sigmoide aplicada a la función lineal w0 + w1*x
    z = w0 + w1 * x
    return 1 / (1 + np.exp(-z))


def graficar_matriz(y_real, y_pred, ax, titulo, etiquetas):
    # Calcula y dibuja la matriz de confusión (filas = clase real, columnas = clase predicha)
    matriz = confusion_matrix(y_real, y_pred)
    sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=[f"Predicho {e}" for e in etiquetas],
                yticklabels=[f"Real {e}" for e in etiquetas])
    ax.set_title(titulo)
    return matriz

---
# Ejercicio 1: tu primer clasificador desde cero

**Enunciado.** Generar un dataset con `make_blobs` de 400 registros, 1 atributo, 2 clases, `cluster_std=2` y `random_state=0`. Luego: (1) mostrar cuántos registros hay de cada clase, (2) dividir en entrenamiento (75 %) y prueba (25 %) estratificando por clase, (3) entrenar una regresión logística, (4) calcular la frontera de decisión, (5) calcular el accuracy en prueba y mostrar la matriz de confusión, y (6) graficar los datos y la curva sigmoide aprendida marcando la frontera.

## Paso 1: generar los datos y contar las clases

`make_blobs` devuelve `X` (los atributos) e `y` (la clase de cada registro). `np.bincount(y)` cuenta cuántas veces aparece cada valor (0 y 1).

In [ ]:
X, y = make_blobs(n_samples=400, centers=2, n_features=1, cluster_std=2, random_state=0)

print("Forma de X:", X.shape, "-> 400 filas y 1 columna")
print("Registros por clase [clase 0, clase 1]:", np.bincount(y))

Hay 200 registros de cada clase: el dataset está **balanceado**. Miremos cómo se distribuyen los datos antes de modelar, con un histograma de `x` separado por clase:

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(x=X.ravel(), hue=y, bins=30, palette=["tab:blue", "tab:red"], alpha=0.6)
plt.title("Ejercicio 1: distribución de x en cada clase")
plt.xlabel("x")
plt.show()

Las dos clases se ubican en zonas distintas de `x`, con una **superposición** en el medio (porque `cluster_std=2` es una dispersión considerable). Eso significa que el modelo va a cometer algunos errores: es esperable.

## Paso 2: dividir en entrenamiento (75 %) y prueba (25 %), estratificando

- `test_size=0.25` → el 25 % de los datos va a prueba.
- `stratify=y` → ambos conjuntos mantienen la **proporción de clases** (50 % / 50 %).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Entrenamiento:", X_train.shape[0], "registros -> por clase:", np.bincount(y_train))
print("Prueba:       ", X_test.shape[0], "registros -> por clase:", np.bincount(y_test))

## Paso 3: entrenar la regresión logística y leer sus parámetros

Aplicamos el flujo de siempre: **instanciar → entrenar (`fit`)**. Después extraemos los parámetros aprendidos:

- `clf.intercept_[0]` → el intercepto `w0`.
- `clf.coef_[0][0]` → el coeficiente `w1`. (`coef_` es una tabla de 2 dimensiones; con 1 atributo y 2 clases tiene 1 fila y 1 columna.)

In [ ]:
clf = LogisticRegression()
clf.fit(X_train, y_train)

w0 = clf.intercept_[0]
w1 = clf.coef_[0][0]

print(f"Intercepto (w0):  {w0:.3f}")
print(f"Coeficiente (w1): {w1:.3f}")
print(f"\nP(clase 1) = 1 / (1 + e^-({w0:.3f} + {w1:.3f} * x))")

## Paso 4: la frontera de decisión

La frontera es el valor de `x` donde la probabilidad vale exactamente 0,5. Eso ocurre cuando `w0 + w1·x = 0`, es decir:

$$ x = -\frac{w_0}{w_1} $$

In [ ]:
frontera = -w0 / w1
print(f"Frontera de decisión: x = {frontera:.3f}")
print(f"Probabilidad de clase 1 justo en la frontera: {sigmoid_function(frontera, w0, w1):.3f}")

La probabilidad en la frontera es **0,5**, como esperábamos. Los registros con `x` menor que ese valor se clasifican como **clase 0** y los de `x` mayor, como **clase 1**.

## Paso 5: accuracy y matriz de confusión en el conjunto de prueba

In [ ]:
y_pred = clf.predict(X_test)

acc_test = accuracy_score(y_test, y_pred)
acc_train = accuracy_score(y_train, clf.predict(X_train))
print(f"Accuracy en entrenamiento: {acc_train:.3f}")
print(f"Accuracy en prueba:        {acc_test:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
matriz = graficar_matriz(y_test, y_pred, ax, "Ejercicio 1: matriz de confusión (prueba)", [0, 1])
plt.show()

VN, FP, FN, VP = matriz.ravel()       # orden: VN, FP, FN, VP
print(f"VN = {VN}, FP = {FP}, FN = {FN}, VP = {VP}")

**Interpretación:**

- El **accuracy ronda el 76 %**: es un resultado **moderado**, bastante menor que el ~98 % que obtuvimos en la práctica. La razón es que acá `cluster_std=2` hace que las clases **se superpongan mucho** y, cuando hay superposición, ningún modelo puede acertar siempre. Un accuracy más bajo **no significa que el modelo esté mal**: refleja que el problema es más difícil.
- Los valores de **entrenamiento y prueba son parecidos**, así que el modelo **generaliza bien** (no hay sobreajuste).
- En la matriz de confusión, la **diagonal** (VN y VP) concentra los aciertos. Los errores no están repartidos del todo parejo: hay **más falsos positivos que falsos negativos** (el modelo se "pasa" más para el lado de la clase 1 que para el de la clase 0). Con un solo conjunto de prueba de 100 registros, esa diferencia puede deberse en parte al azar del sorteo.

## Paso 6: graficar los datos, la sigmoide y la frontera

In [ ]:
x_plot = np.linspace(X.min() - 1, X.max() + 1, 300)

plt.figure(figsize=(12, 6))
plt.scatter(X_test, y_test, c=y_test, alpha=0.6, lw=0.5, edgecolor="grey", cmap=cm.coolwarm, label="Datos de prueba")
plt.plot(x_plot, sigmoid_function(x_plot, w0, w1), color="green", linewidth=2.5, label="Curva sigmoide aprendida")
plt.axhline(0.5, color="gray", linestyle="--", label="Umbral = 0,5")
plt.axvline(frontera, color="black", linestyle="--", label=f"Frontera (x = {frontera:.2f})")
plt.xlabel("x")
plt.ylabel("Probabilidad de clase 1")
plt.title("Ejercicio 1: regresión logística con un atributo")
plt.legend()
plt.show()

La curva sigmoide pasa de 0 a 1 justo en la zona donde las clases se mezclan, y la línea negra marca la frontera que corresponde al 50 %. Comparada con la de la práctica (con `cluster_std=1.5`), esta transición es **más suave** porque hay más superposición entre las clases: el modelo está **menos seguro** en la zona intermedia.

---
# Ejercicio 2: elegir el umbral

**Enunciado.** Suponer un modelo que detecta una **enfermedad grave**, donde un falso negativo es mucho peor que un falso positivo. (1) Calcular precision, recall y F1 para umbrales de 0.1, 0.3, 0.5 y 0.7. (2) Mostrar la matriz de confusión para el umbral 0.5 y para el 0.1. (3) Elegir un umbral y justificar.

## Paso 1: preparar el modelo

Usamos el dataset de la sección 5.5 de la práctica, donde las clases **se superponen bastante** (así hay errores para analizar). Lo reconstruimos igual: dos nubes centradas en 0 y en 2,5, con mucha dispersión.

Lo importante en este ejercicio es **no re-entrenar el modelo para cada umbral**: entrenamos **una sola vez** y obtenemos las **probabilidades** con `predict_proba`. Después solo cambiamos **cómo convertimos la probabilidad en clase**.

In [ ]:
X_s, y_s = make_blobs(n_samples=1000, centers=[[0], [2.5]], cluster_std=1.8, random_state=3)
Xs_train, Xs_test, ys_train, ys_test = train_test_split(X_s, y_s, test_size=0.3, random_state=42, stratify=y_s)

clf_s = LogisticRegression().fit(Xs_train, ys_train)
proba_s = clf_s.predict_proba(Xs_test)[:, 1]       # probabilidad de clase 1 para cada registro de prueba

print("Primeras 5 probabilidades de clase 1:", proba_s[:5].round(3))

## Paso 2: métricas para cada umbral

Recorremos la lista de umbrales con un bucle `for`. En cada vuelta:

1. `proba_s >= umbral` compara cada probabilidad con el umbral y da `True`/`False`.
2. `.astype(int)` convierte `True`/`False` en `1`/`0`: esas son las **clases predichas**.
3. Calculamos precision, recall y F1, y también contamos los **falsos negativos** (los casos graves que se nos escapan) y los **falsos positivos** (las falsas alarmas).

In [ ]:
filas = []
for umbral in [0.1, 0.3, 0.5, 0.7]:
    pred = (proba_s >= umbral).astype(int)               # aplicamos el umbral
    VN, FP, FN, VP = confusion_matrix(ys_test, pred).ravel()
    filas.append({
        "Umbral": umbral,
        "Precision": precision_score(ys_test, pred),
        "Recall": recall_score(ys_test, pred),
        "F1": f1_score(ys_test, pred),
        "Falsos negativos (FN)": FN,
        "Falsos positivos (FP)": FP,
    })

tabla_umbrales = pd.DataFrame(filas)
tabla_umbrales.round(3)

Mirá la tendencia a medida que **sube el umbral**:

- El **recall baja** y los **falsos negativos aumentan**: el modelo exige más evidencia para decir "positivo", así que se le escapan más casos reales.
- La **precision sube** y los **falsos positivos disminuyen**: lo que marca como positivo es más confiable, pero detecta menos.

Es el **compromiso (*trade-off*) entre precision y recall**: no se pueden maximizar las dos a la vez.

## Paso 3: matrices de confusión para los umbrales 0.5 y 0.1

Dibujamos las dos matrices lado a lado (`plt.subplots(1, 2, ...)`) para compararlas visualmente.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 5))

for eje, umbral in zip(ejes, [0.5, 0.1]):               # zip recorre eje y umbral "en paralelo"
    pred = (proba_s >= umbral).astype(int)
    graficar_matriz(ys_test, pred, eje, f"Umbral = {umbral}", [0, 1])

plt.tight_layout()
plt.show()

**Cómo comparar las matrices:** fijate en la fila **"Real 1"** (los enfermos). Con el umbral 0.5, una parte de ellos queda en "Predicho 0" (son los **falsos negativos**). Con el umbral 0.1, casi todos pasan a "Predicho 1" (los falsos negativos prácticamente desaparecen).

El precio es que, en la fila **"Real 0"** (los sanos), **muchos más** caen en "Predicho 1": aparecen **muchos falsos positivos**.

## Paso 4: ¿qué umbral elegir?

El enunciado dice que **un falso negativo es mucho peor que un falso positivo** (dejar sin detectar a una persona con una enfermedad grave es peor que una falsa alarma). Entonces el objetivo es **maximizar el recall**, aceptando perder precision.

> ✅ **Respuesta:** elegiría un umbral **bajo** (por ejemplo **0.1**, o 0.3 si se quiere un punto intermedio). Con 0.1 el recall es casi perfecto, es decir, **casi ningún enfermo se nos escapa**. A cambio hay muchas falsas alarmas, pero en este contexto se resuelven con un **estudio de confirmación**, mucho menos costoso que no detectar la enfermedad.
>
> En la práctica, esa decisión no la toma solo quien modela: se **discute con especialistas** del área (por ejemplo, el equipo médico), que saben cuánto cuesta cada tipo de error.

Comprobamos con código cuántos falsos negativos evita el umbral bajo:

In [ ]:
fn_05 = tabla_umbrales.loc[tabla_umbrales["Umbral"] == 0.5, "Falsos negativos (FN)"].iloc[0]
fn_01 = tabla_umbrales.loc[tabla_umbrales["Umbral"] == 0.1, "Falsos negativos (FN)"].iloc[0]
fp_05 = tabla_umbrales.loc[tabla_umbrales["Umbral"] == 0.5, "Falsos positivos (FP)"].iloc[0]
fp_01 = tabla_umbrales.loc[tabla_umbrales["Umbral"] == 0.1, "Falsos positivos (FP)"].iloc[0]

print(f"Umbral 0.5 -> {fn_05} falsos negativos y {fp_05} falsos positivos")
print(f"Umbral 0.1 -> {fn_01} falsos negativos y {fp_01} falsos positivos")
print(f"\nBajar el umbral evita {fn_05 - fn_01} casos graves sin detectar, a costa de {fp_01 - fp_05} falsas alarmas más.")

---
# Ejercicio 3: la trampa del accuracy

**Enunciado.** Generar un dataset muy desbalanceado con `make_classification` (3000 registros, 97 % / 3 %). (1) Dividir en entrenamiento y prueba (70 / 30, estratificado). (2) Calcular el accuracy de un "modelo tonto" que siempre predice 0. (3) Entrenar una regresión logística y calcular su accuracy, recall y F1 para la clase 1. (4) Mostrar la matriz de confusión: ¿cuántos positivos reales detectó? (5) Bajar el umbral a 0.2 y ver qué pasa con recall y precision.

## Paso 1: generar el dataset desbalanceado

Los parámetros de `make_classification`:

| Parámetro | Significado |
|---|---|
| `n_samples=3000` | Cantidad de registros |
| `n_features=2`, `n_informative=2`, `n_redundant=0` | 2 atributos, ambos útiles, ninguno redundante |
| `n_clusters_per_class=1` | Una sola "nube" por clase |
| `weights=[0.97, 0.03]` | **97 %** de registros de clase 0 y **3 %** de clase 1 |
| `class_sep=1.2` | Qué tan separadas están las clases (más alto = más fácil) |

In [ ]:
X_d, y_d = make_classification(n_samples=3000, n_features=2, n_informative=2, n_redundant=0,
                               n_clusters_per_class=1, weights=[0.97, 0.03],
                               class_sep=1.2, random_state=1)

print("Registros por clase [clase 0, clase 1]:", np.bincount(y_d))
print(f"Porcentaje de clase 1: {y_d.mean():.1%}")

plt.figure(figsize=(9, 6))
plt.scatter(X_d[:, 0], X_d[:, 1], c=y_d, cmap=cm.coolwarm, alpha=0.6, edgecolor="grey", lw=0.3)
plt.title("Ejercicio 3: dataset desbalanceado (rojo = clase 1, la minoritaria)")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

En el gráfico se ve que **los puntos rojos (clase 1) son muy pocos** frente a los azules. Esa es justamente la dificultad de este tipo de problemas.

## Paso 2: dividir (70 % / 30 %, estratificado)

Con `stratify=y_d` nos aseguramos de que **tanto el conjunto de entrenamiento como el de prueba conserven el 3 %** de casos positivos. Sin eso, por mala suerte, el conjunto de prueba podría quedarse con muy pocos positivos (o ninguno).

In [ ]:
Xd_train, Xd_test, yd_train, yd_test = train_test_split(X_d, y_d, test_size=0.3, random_state=42, stratify=y_d)

print("Entrenamiento:", np.bincount(yd_train), "-> positivos:", f"{yd_train.mean():.1%}")
print("Prueba:       ", np.bincount(yd_test), "-> positivos:", f"{yd_test.mean():.1%}")

## Paso 3: el "modelo tonto" que siempre predice 0

`np.zeros_like(yd_test)` crea un arreglo de **ceros del mismo tamaño** que `yd_test`: es decir, un "modelo" que **siempre responde clase 0**, sin mirar los datos.

Calculamos su accuracy y también su recall, para ver qué pasa con la clase que nos interesa detectar.

In [ ]:
pred_tonto = np.zeros_like(yd_test)

print(f"Accuracy del modelo tonto: {accuracy_score(yd_test, pred_tonto):.3f}")
print(f"Recall del modelo tonto (clase 1): {recall_score(yd_test, pred_tonto):.3f}")

**¡Un accuracy altísimo sin haber aprendido nada!** El modelo tonto acierta casi siempre simplemente porque **casi todos los registros son de clase 0**. Pero su recall es **0**: **no detecta ni un solo caso positivo**. Esa es la trampa del accuracy: un número que "se ve bien" y esconde un modelo inútil.

## Paso 4: la regresión logística real

In [ ]:
clf_d = LogisticRegression().fit(Xd_train, yd_train)
pred_real = clf_d.predict(Xd_test)

comparacion = pd.DataFrame({
    "Modelo tonto (siempre 0)": {
        "Accuracy": accuracy_score(yd_test, pred_tonto),
        "Precision (clase 1)": precision_score(yd_test, pred_tonto, zero_division=0),   # zero_division evita un aviso (no hay positivos predichos)
        "Recall (clase 1)": recall_score(yd_test, pred_tonto),
        "F1 (clase 1)": f1_score(yd_test, pred_tonto),
    },
    "Regresión logística (umbral 0.5)": {
        "Accuracy": accuracy_score(yd_test, pred_real),
        "Precision (clase 1)": precision_score(yd_test, pred_real),
        "Recall (clase 1)": recall_score(yd_test, pred_real),
        "F1 (clase 1)": f1_score(yd_test, pred_real),
    },
}).T

comparacion.round(3)

Observá que los accuracy de los dos modelos son **parecidos** (ambos muy altos), pero el **recall y el F1 de la clase 1** muestran la diferencia real: el modelo tonto tiene 0, mientras que la regresión logística **sí detecta** una parte de los positivos.

## Paso 5: matriz de confusión del modelo

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
matriz = graficar_matriz(yd_test, pred_real, ax, "Ejercicio 3: matriz de confusión (umbral 0.5)", [0, 1])
plt.show()

VN, FP, FN, VP = matriz.ravel()
total_positivos = VP + FN
print(f"VN = {VN}, FP = {FP}, FN = {FN}, VP = {VP}")
print(f"\nPositivos reales en prueba: {total_positivos}")
print(f"Positivos detectados:       {VP}  ({VP / total_positivos:.1%})")
print(f"Positivos que se escaparon: {FN}  ({FN / total_positivos:.1%})")

Con el umbral por defecto, el modelo **detecta solo una parte de los positivos reales**; el resto se le escapa (falsos negativos). Es decir, aun con un accuracy altísimo, **no detecta bien la clase minoritaria**: suele pasar porque, al haber tan pocos ejemplos positivos, el modelo "prefiere" decir "negativo" ante la duda.

## Paso 6: bajar el umbral a 0.2

Obtenemos las probabilidades con `predict_proba` y las convertimos en clases con un umbral más bajo. Así el modelo **se anima más** a decir "positivo". Comparamos los dos umbrales.

In [ ]:
proba_d = clf_d.predict_proba(Xd_test)[:, 1]

filas = []
for umbral in [0.5, 0.2]:
    pred = (proba_d >= umbral).astype(int)
    VN, FP, FN, VP = confusion_matrix(yd_test, pred).ravel()
    filas.append({
        "Umbral": umbral,
        "Accuracy": accuracy_score(yd_test, pred),
        "Precision": precision_score(yd_test, pred),
        "Recall": recall_score(yd_test, pred),
        "F1": f1_score(yd_test, pred),
        "FN": FN,
        "FP": FP,
    })

pd.DataFrame(filas).round(3)

## Conclusiones del ejercicio 3

1. **El accuracy engaña** cuando hay clases desbalanceadas: un modelo que no aprende nada ("siempre 0") consigue un accuracy altísimo.
2. Para evaluar de verdad hay que mirar las métricas **de la clase que nos interesa**: **recall, precision y F1** de la clase 1, y la **matriz de confusión**.
3. **Bajar el umbral a 0.2** hace que el **recall suba** (se detectan más positivos y hay menos falsos negativos), a costa de que la **precision baje** (aparecen más falsos positivos). Es el mismo compromiso del ejercicio 2.
4. Existen otras herramientas para este tipo de problemas: el parámetro `class_weight="balanced"` de `LogisticRegression` (que le da más peso a la clase minoritaria durante el entrenamiento), o técnicas de *remuestreo* (sobremuestrear la clase minoritaria o submuestrear la mayoritaria).

> 💡 **Para probar:** reemplazá `LogisticRegression()` por `LogisticRegression(class_weight="balanced")` y volvé a ejecutar. Vas a ver que el recall sube mucho de entrada (de ≈ 0,67 a ≈ 0,90), sin tocar el umbral, pero con un costo: la precision baja bastante (de ≈ 0,91 a ≈ 0,36). Es otra forma de moverse en el mismo compromiso entre precision y recall.

---
# Ejercicio 4 (desafío): clasificación multiclase

**Enunciado.** Generar un dataset con `make_blobs(n_samples=900, centers=[[0, 0], [5, 0], [2.5, 4]], cluster_std=1.5, random_state=2)` (3 clases, 2 atributos). (1) Dividir (70 / 30, estratificado) y entrenar una regresión logística. (2) Calcular el accuracy y mostrar la matriz de confusión de 3×3: ¿entre qué clases hay más confusiones? (3) Imprimir el `classification_report`. (4) Elegir un punto nuevo y mostrar las probabilidades de las tres clases: ¿suman 1? (5) Graficar las regiones de decisión.

## Paso 1: generar los datos y mirarlos

Con `centers=[[0, 0], [5, 0], [2.5, 4]]` indicamos **tres centros**: uno por cada clase, formando aproximadamente un triángulo.

In [ ]:
X4, y4 = make_blobs(n_samples=900, centers=[[0, 0], [5, 0], [2.5, 4]], cluster_std=1.5, random_state=2)

print("Forma de X4:", X4.shape)
print("Registros por clase [clase 0, clase 1, clase 2]:", np.bincount(y4))

plt.figure(figsize=(9, 6))
plt.scatter(X4[:, 0], X4[:, 1], c=y4, cmap="viridis", alpha=0.6, edgecolor="grey", lw=0.3)
plt.title("Ejercicio 4: tres clases con dos atributos")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

Se distinguen tres nubes, con algo de superposición en los bordes entre ellas.

## Paso 2: dividir, entrenar y evaluar

**El código es exactamente el mismo que para dos clases**: Scikit-Learn detecta automáticamente que hay tres clases y usa la generalización de la sigmoide llamada **softmax**.

In [ ]:
X4_train, X4_test, y4_train, y4_test = train_test_split(X4, y4, test_size=0.3, random_state=42, stratify=y4)

clf4 = LogisticRegression()
clf4.fit(X4_train, y4_train)
y4_pred = clf4.predict(X4_test)

print("Clases que conoce el modelo:", clf4.classes_)
print("Forma de coef_:", clf4.coef_.shape, "-> una fila de coeficientes por cada clase, una columna por atributo")
print(f"\nAccuracy en entrenamiento: {accuracy_score(y4_train, clf4.predict(X4_train)):.3f}")
print(f"Accuracy en prueba:        {accuracy_score(y4_test, y4_pred):.3f}")

## Paso 3: matriz de confusión de 3×3

Con tres clases, la matriz tiene **3 filas y 3 columnas**. Se lee igual que antes: las **filas** son la clase real, las **columnas** la clase predicha, y los **aciertos están en la diagonal**.

Además de dibujarla, buscamos por código **dónde están los errores**: ponemos en 0 la diagonal (con `np.fill_diagonal`) y buscamos el valor máximo que queda, que es el par de clases que más se confunden.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
matriz4 = graficar_matriz(y4_test, y4_pred, ax, "Ejercicio 4: matriz de confusión (3 clases)", [0, 1, 2])
plt.show()

# Buscamos el par de clases con más confusiones: anulamos la diagonal (aciertos) y vemos el máximo
errores = matriz4.copy()
np.fill_diagonal(errores, 0)
real, predicha = np.unravel_index(errores.argmax(), errores.shape)     # posición (fila, columna) del máximo

print("Matriz de errores (sin la diagonal):")
print(errores)
print(f"\nLa confusión más frecuente: clase real {real} clasificada como clase {predicha} ({errores.max()} casos).")
print(f"Total de errores: {errores.sum()} de {len(y4_test)} registros de prueba.")

**Interpretación:** el modelo acierta cerca del 91 % y los errores son **pocos y repartidos entre todos los pares de clases**. Tiene sentido: los tres centros forman un triángulo casi equilátero (a unas 4,7 a 5 unidades uno de otro) y la dispersión de cada nube (1,5) hace que **todas se rocen en los bordes**. Aun así, el par que más se confunde es el de las clases 0 y 2 (11 errores sumando ambos sentidos), y los otros pares tienen unos 6 errores cada uno.

> 📝 Los números exactos dependen del sorteo; lo importante es la **idea**: las confusiones aparecen entre clases cuyas nubes están **cerca y se superponen**. Si alejaras mucho los centros, casi desaparecerían.

## Paso 4: el reporte de clasificación

`classification_report` calcula precision, recall y F1 **para cada clase por separado** (`support` es la cantidad de registros reales de cada clase). Con varias clases, aparecen además dos promedios: **macro avg** (promedio simple de las clases) y **weighted avg** (promedio ponderado según el tamaño de cada clase).

In [ ]:
print(classification_report(y4_test, y4_pred, target_names=["Clase 0", "Clase 1", "Clase 2"]))

## Paso 5: probabilidades para un punto nuevo

Elegimos el punto `[2.5, 1.5]`, que está **en el medio** del triángulo formado por los tres centros. Se espera que el modelo esté **menos seguro** ahí. Le pasamos el punto como una **tabla de 1 fila y 2 columnas** (`[[2.5, 1.5]]`, con doble corchete).

In [ ]:
punto_nuevo = np.array([[2.5, 1.5]])

probabilidades = clf4.predict_proba(punto_nuevo)[0]         # 3 valores: uno por clase
clase_predicha = clf4.predict(punto_nuevo)[0]

for clase, p in enumerate(probabilidades):
    print(f"P(clase {clase}) = {p:.3f}  ({p:.1%})")

print(f"\nSuma de las tres probabilidades: {probabilidades.sum():.3f}")
print(f"Clase predicha (la de mayor probabilidad): {clase_predicha}")

**Sí, las tres probabilidades suman 1**: es una propiedad de la función softmax (reparte el 100 % entre todas las clases). La clase predicha es simplemente **la de mayor probabilidad**.

Fijate también que, al ser un punto "intermedio", **ninguna probabilidad es cercana a 1**: el modelo expresa **duda**. Esa información (que `predict` por sí solo no muestra) es lo que hace tan útil a `predict_proba`.

## Paso 6: regiones de decisión

Usamos la misma técnica de la práctica: armamos una **grilla** de puntos que cubra el plano (`np.meshgrid`), le pedimos al modelo la clase de cada punto y **coloreamos el fondo** con `plt.contourf`. Encima dibujamos los datos de prueba y marcamos el punto nuevo con una estrella.

In [ ]:
x_min, x_max = X4[:, 0].min() - 1, X4[:, 0].max() + 1
y_min, y_max = X4[:, 1].min() - 1, X4[:, 1].max() + 1

g1, g2 = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grilla = np.column_stack([g1.ravel(), g2.ravel()])                  # tabla con todos los puntos de la grilla
clases_grilla = clf4.predict(grilla).reshape(g1.shape)              # clase predicha en cada punto

plt.figure(figsize=(9, 6))
plt.contourf(g1, g2, clases_grilla, alpha=0.25, cmap="viridis", levels=[-0.5, 0.5, 1.5, 2.5])
plt.scatter(X4_test[:, 0], X4_test[:, 1], c=y4_test, cmap="viridis", edgecolor="grey", lw=0.5, alpha=0.8)
plt.scatter(punto_nuevo[:, 0], punto_nuevo[:, 1], marker="*", s=350, color="red", edgecolor="black", label="Punto nuevo [2.5, 1.5]")
plt.title("Ejercicio 4: regiones de decisión con 3 clases")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.show()

El plano queda dividido en **tres regiones separadas por fronteras rectas** (el modelo sigue siendo lineal). La estrella roja cae cerca de donde las tres regiones se encuentran: por eso las probabilidades del paso anterior estaban repartidas.

## Conclusiones del ejercicio 4

- La regresión logística se extiende a **más de dos clases usando exactamente el mismo código**.
- `predict_proba` devuelve **una columna por clase** y cada fila **suma 1**; `coef_` tiene **una fila por clase**.
- Las **confusiones** aparecen donde las nubes de distintas clases **se superponen**, es decir, en los bordes entre clases cercanas.
- Las **fronteras siguen siendo rectas**: si las clases estuvieran dispuestas de otra manera (por ejemplo, anillos), harían falta atributos polinómicos u otro algoritmo.

---
# 🎓 Resumen de lo que practicaste

| Ejercicio | Qué aprendiste |
|---|---|
| **1** | El flujo completo de clasificación: generar datos → dividir (estratificando) → entrenar → hallar la frontera `x = -w0/w1` → evaluar con matriz de confusión |
| **2** | El **umbral** es una decisión de negocio: bajarlo aumenta el recall (menos falsos negativos) y baja la precision (más falsas alarmas) |
| **3** | Con clases desbalanceadas el **accuracy engaña**; hay que mirar recall, precision, F1 y la matriz de confusión de la clase minoritaria |
| **4** | La regresión logística **se extiende a varias clases** con el mismo código; las probabilidades suman 1 y las confusiones ocurren entre clases cercanas |

**¡Buen trabajo!** 🎉